In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🤖 Machine Learning Model Serving Guide

**Production ML model serving architectures, inference optimization, and monitoring**

This guide covers:
- Model serving patterns
- Inference optimization
- Batch vs real-time serving
- Model versioning
- A/B testing frameworks
- Model monitoring
- Performance metrics
- Automated retraining
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Model Serving Architecture

### Model Server Implementation
```python
from dataclasses import dataclass, field
from typing import Dict, Optional, List, Any
from datetime import datetime
from enum import Enum
import json

class ModelFormat(Enum):
    TENSORFLOW = "tensorflow"
    PYTORCH = "pytorch"
    ONNX = "onnx"
    SKLEARN = "sklearn"

@dataclass
class ModelMetadata:
    """Model metadata"""
    model_id: str
    name: str
    version: str
    format: ModelFormat
    input_schema: Dict
    output_schema: Dict
    performance_metrics: Dict = field(default_factory=dict)
    created_at: datetime = field(default_factory=datetime.utcnow)

class ModelRegistry:
    """Central model registry"""
    
    def __init__(self):
        self.models: Dict[str, List[ModelMetadata]] = {}
        self.active_models: Dict[str, ModelMetadata] = {}
    
    def register_model(self, metadata: ModelMetadata) -> bool:
        """Register model"""
        
        if metadata.name not in self.models:
            self.models[metadata.name] = []
        
        self.models[metadata.name].append(metadata)
        
        return True
    
    def get_model_version(self, model_name: str, version: str) -> Optional[ModelMetadata]:
        """Get specific model version"""
        
        if model_name not in self.models:
            return None
        
        for m in self.models[model_name]:
            if m.version == version:
                return m
        
        return None
    
    def get_latest_model(self, model_name: str) -> Optional[ModelMetadata]:
        """Get latest model version"""
        
        if model_name not in self.models or not self.models[model_name]:
            return None
        
        return sorted(self.models[model_name], 
                     key=lambda m: m.created_at, reverse=True)[0]
    
    def set_active_model(self, model_name: str, version: str) -> bool:
        """Set active model version"""
        
        model = self.get_model_version(model_name, version)
        
        if not model:
            return False
        
        self.active_models[model_name] = model
        
        return True
    
    def get_active_model(self, model_name: str) -> Optional[ModelMetadata]:
        """Get active model"""
        
        return self.active_models.get(model_name)

class ModelServer:
    """Serve models for inference"""
    
    def __init__(self, registry: ModelRegistry):
        self.registry = registry
        self.inference_cache: Dict[str, Dict] = {}
        self.metrics: List[Dict] = []
    
    def predict(self, model_name: str, features: Dict) -> Dict:
        """Make prediction"""
        
        model = self.registry.get_active_model(model_name)
        
        if not model:
            raise Exception(f"Model {model_name} not found")
        
        # Record inference request
        inference_id = f"{model_name}_{datetime.utcnow().timestamp()}"
        
        # Cache key
        cache_key = json.dumps(features, sort_keys=True)
        
        # Check cache
        if cache_key in self.inference_cache:
            prediction = self.inference_cache[cache_key]
            prediction['cached'] = True
            return prediction
        
        # Run inference (mock)
        prediction = {
            'model_id': model.model_id,
            'version': model.version,
            'inference_id': inference_id,
            'features': features,
            'prediction': self._run_model(model, features),
            'confidence': 0.95,
            'timestamp': datetime.utcnow().isoformat(),
            'cached': False
        }
        
        # Cache result
        self.inference_cache[cache_key] = prediction
        
        # Record metrics
        self.metrics.append({
            'model_name': model_name,
            'timestamp': datetime.utcnow().isoformat(),
            'inference_time_ms': 25.5,
            'cache_hit': False
        })
        
        return prediction
    
    def _run_model(self, model: ModelMetadata, features: Dict) -> Dict:
        """Run model inference"""
        
        # Mock inference
        return {'prediction': 0.75, 'class': 'positive'}
    
    def batch_predict(self, model_name: str, features_list: List[Dict]) -> List[Dict]:
        """Batch prediction"""
        
        predictions = []
        
        for features in features_list:
            predictions.append(self.predict(model_name, features))
        
        return predictions
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. A/B Testing & Canary Deployment

### Experiment Framework
```python
class Experiment:
    """A/B test experiment"""
    
    def __init__(self, exp_id: str, control_model: str, treatment_model: str, 
                 split_percentage: float = 0.5):
        self.exp_id = exp_id
        self.control_model = control_model
        self.treatment_model = treatment_model
        self.split_percentage = split_percentage
        self.start_time = datetime.utcnow()
        self.results: List[Dict] = []
    
    def select_variant(self, user_id: str) -> str:
        """Select variant for user"""
        
        # Hash user to consistent variant
        hash_value = hash(user_id) % 100
        
        if hash_value < (self.split_percentage * 100):
            return self.treatment_model
        
        return self.control_model
    
    def record_result(self, user_id: str, model: str, prediction: Dict, 
                     actual: Any = None):
        """Record experiment result"""
        
        self.results.append({
            'user_id': user_id,
            'model': model,
            'prediction': prediction,
            'actual': actual,
            'timestamp': datetime.utcnow().isoformat()
        })
    
    def get_metrics(self) -> Dict:
        """Get experiment metrics"""
        
        control_results = [r for r in self.results if r['model'] == self.control_model]
        treatment_results = [r for r in self.results if r['model'] == self.treatment_model]
        
        return {
            'control_count': len(control_results),
            'treatment_count': len(treatment_results),
            'control_accuracy': self._calculate_accuracy(control_results),
            'treatment_accuracy': self._calculate_accuracy(treatment_results),
            'improvement': self._calculate_accuracy(treatment_results) - self._calculate_accuracy(control_results)
        }
    
    def _calculate_accuracy(self, results: List[Dict]) -> float:
        """Calculate accuracy"""
        
        if not results:
            return 0.0
        
        correct = sum(1 for r in results if r.get('actual') is not None)
        
        return correct / len(results) if results else 0.0

class CanaryDeployment:
    """Canary deployment strategy"""
    
    def __init__(self, current_model: str, new_model: str):
        self.current_model = current_model
        self.new_model = new_model
        self.traffic_split = 0.0  # Start with 0% new model
        self.error_rate_threshold = 0.05
        self.start_time = datetime.utcnow()
    
    def get_model_for_request(self, request_id: str) -> str:
        """Select model based on canary traffic split"""
        
        hash_value = hash(request_id) % 100
        
        if hash_value < self.traffic_split:
            return self.new_model
        
        return self.current_model
    
    def increase_traffic(self, percentage: float) -> bool:
        """Gradually increase traffic to new model"""
        
        if self.traffic_split + percentage > 100:
            self.traffic_split = 100
        else:
            self.traffic_split += percentage
        
        return True
    
    def rollback(self) -> bool:
        """Rollback to current model"""
        
        self.traffic_split = 0.0
        
        return True
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Model Monitoring & Metrics

### Performance Monitoring
```python
class ModelMonitor:
    """Monitor model performance"""
    
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.performance_data: List[Dict] = []
        self.alerts: List[Dict] = []
    
    def record_prediction(self, prediction: Dict, actual: Any = None, 
                         latency_ms: float = 0):
        """Record prediction"""
        
        self.performance_data.append({
            'timestamp': datetime.utcnow().isoformat(),
            'prediction': prediction,
            'actual': actual,
            'latency_ms': latency_ms,
            'correct': prediction.get('prediction') == actual if actual else None
        })
    
    def get_accuracy(self, window_size: int = 100) -> float:
        """Get recent accuracy"""
        
        recent = self.performance_data[-window_size:]
        correct = sum(1 for r in recent if r.get('correct'))
        
        return correct / len(recent) if recent else 0.0
    
    def get_average_latency(self, window_size: int = 100) -> float:
        """Get average latency"""
        
        recent = self.performance_data[-window_size:]
        latencies = [r['latency_ms'] for r in recent]
        
        return sum(latencies) / len(latencies) if latencies else 0.0
    
    def detect_drift(self) -> bool:
        """Detect data drift"""
        
        if len(self.performance_data) < 100:
            return False
        
        # Compare recent accuracy to baseline
        recent_accuracy = self.get_accuracy(50)
        older_accuracy = self.get_accuracy(window_size=100)[-50:]
        
        # Flag if accuracy dropped significantly
        if recent_accuracy < older_accuracy * 0.8:
            self.alerts.append({
                'type': 'data_drift',
                'timestamp': datetime.utcnow().isoformat(),
                'recent_accuracy': recent_accuracy,
                'baseline_accuracy': older_accuracy
            })
            
            return True
        
        return False
    
    def get_health_score(self) -> Dict:
        """Get model health score"""
        
        accuracy = self.get_accuracy()
        latency = self.get_average_latency()
        
        # Normalize scores 0-100
        accuracy_score = accuracy * 100
        latency_score = max(0, 100 - (latency / 10))  # 100ms = 0 points
        
        overall = (accuracy_score + latency_score) / 2
        
        return {
            'overall_score': overall,
            'accuracy_score': accuracy_score,
            'latency_score': latency_score,
            'status': 'healthy' if overall > 80 else 'degraded' if overall > 60 else 'unhealthy'
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. ML Serving Checklist

✅ **Model Registry**
- [ ] Registry implemented
- [ ] Versioning tracked
- [ ] Metadata stored
- [ ] Active model management
- [ ] Easy rollback

✅ **Inference**
- [ ] Model loading
- [ ] Batch processing
- [ ] Caching enabled
- [ ] Latency optimized
- [ ] GPU utilized

✅ **Experimentation**
- [ ] A/B testing framework
- [ ] Statistical significance
- [ ] Experiment tracking
- [ ] Automatic rollout
- [ ] Metrics calculation

✅ **Monitoring**
- [ ] Performance tracking
- [ ] Data drift detection
- [ ] Anomalies flagged
- [ ] Health scores
- [ ] Alerting active

✅ **Operational**
- [ ] Automation
- [ ] Documentation
- [ ] Team training
- [ ] Disaster recovery
- [ ] Cost monitoring
</VSCode.Cell>
```